# Extractive Question Answering on SQuAD — Fine-tuning DistilBERT

This notebook fine-tunes a pretrained **DistilBERT** encoder on the **SQuAD** dataset
for extractive question answering: given a passage and a question, the model predicts
the start and end token positions of the answer span inside the passage.

**Run this notebook top to bottom in Google Colab** (Runtime → Change runtime type → GPU).
The first code cells install libraries and download the dataset + pretrained weights from
the Hugging Face Hub, which requires an internet connection.

Pipeline:
1. Install & import libraries
2. Load and explore the SQuAD dataset
3. Tokenize question/context pairs and convert character-level answer spans to token
   start/end positions (using the tokenizer's offset mapping), handling long contexts
   with a sliding window (stride)
4. Fine-tune `distilbert-base-uncased` with a QA head
5. Post-process model logits back into predicted answer text spans
6. Evaluate with Exact Match (EM) and F1 on the validation set
7. Look at failure cases
8. Save the fine-tuned model + tokenizer for the Streamlit app


## 1. Install & import libraries

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate torch


In [ ]:
import numpy as np
import torch
import collections
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## 2. Load and explore the dataset

SQuAD has ~87,600 training examples and ~10,600 validation examples. Each example has
a `context` (paragraph), a `question`, and `answers` — a dict with the literal answer
text(s) and their starting character offset(s) inside the context. The answer is always
a span lifted verbatim from the context (that's what makes this *extractive* QA), so the
model's job is to predict where the span starts and ends, not to generate free text.


In [ ]:
raw_datasets = load_dataset("rajpurkar/squad")
raw_datasets


In [ ]:
# Look at a handful of raw examples before writing any preprocessing code.
for i in range(3):
    ex = raw_datasets["train"][i]
    print("CONTEXT: ", ex["context"][:300], "...")
    print("QUESTION:", ex["question"])
    print("ANSWER:  ", ex["answers"])
    print("-" * 80)


## 3. Preprocessing: turn character spans into token start/end positions

Two things make this fiddly:

- **Truncation + stride.** Some contexts are longer than the model's max sequence
  length. We truncate only the *context* (`truncation="only_second"`) and use a
  sliding **stride** so long contexts are split into overlapping windows instead of
  losing content — each window becomes its own training feature.
- **Offset mapping.** The tokenizer's `return_offsets_mapping=True` gives us, for every
  token, the (start_char, end_char) span it covers in the original text. We use that to
  find which *tokens* correspond to the answer's character span. If the answer falls
  outside the current window (possible after truncation/striding), we label that
  feature's start/end position as the `[CLS]` token index, signalling "no answer in this
  window".


In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 384      # max tokens per feature (question + context window)
DOC_STRIDE = 128      # overlap between consecutive windows of a long context

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
assert tokenizer.is_fast, "This notebook needs a fast tokenizer for offset mapping."


In [ ]:
def prepare_train_features(examples):
    # Tokenize with truncation on the context only, a stride for long contexts,
    # and offsets so we can map character positions -> token positions.
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            # No answer for this example (not the case for SQuAD v1, but keep it robust)
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Find where the context (sequence_id == 1) starts and ends in this feature.
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # If the answer isn't fully inside this window, label it with the CLS index.
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # Otherwise walk inwards from both ends to find the exact token boundaries.
            idx = token_start_index
            while idx <= token_end_index and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = token_end_index
            while idx >= token_start_index and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


In [ ]:
# For validation we keep the offset mapping (needed later to pull the predicted text
# back out of the original context) and an example_id to group features by example.
def prepare_validation_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        example_ids.append(examples["id"][sample_index])

        # Mask out offsets that don't belong to the context (question tokens, specials)
        # so the postprocessing step only ever looks at context spans.
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    tokenized["example_id"] = example_ids
    return tokenized


In [ ]:
# OPTIONAL: for a faster first run, subsample the training set. Set to None to use
# the full ~87.6k training examples (recommended for your final submitted run).
TRAIN_SUBSET_SIZE = None   # e.g. 10000 for a quick smoke-test run

train_dataset_raw = raw_datasets["train"]
if TRAIN_SUBSET_SIZE is not None:
    train_dataset_raw = train_dataset_raw.shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))

train_dataset = train_dataset_raw.map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

validation_dataset = raw_datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

print(len(train_dataset_raw), "raw training examples ->", len(train_dataset), "training features")
print(len(raw_datasets["validation"]), "raw validation examples ->", len(validation_dataset), "validation features")


## 4. Fine-tune DistilBERT with a question-answering head

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT).to(device)


In [ ]:
args = TrainingArguments(
    output_dir="squad-distilbert-checkpoints",
    eval_strategy="no",        # we evaluate manually with the SQuAD metric below
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# Trainer needs plain tensors; the validation set carries extra columns
# (example_id, offset_mapping) used only for postprocessing, so we pass a
# stripped-down version to the Trainer for the training loop itself.
validation_dataset_for_model = validation_dataset.remove_columns(["example_id", "offset_mapping"])

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset_for_model,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)


In [ ]:
train_result = trainer.train()
train_result


## 5. Post-process predictions into answer text spans

The model outputs start/end logits over token positions for every feature. To get an
actual answer string per *example* (an example can span multiple features/windows if
its context was long), we:

1. Take the top `n_best_size` start and end logit positions per feature.
2. Keep only combinations where start <= end and the span length is reasonable.
3. Convert token positions back to character positions using the saved offset mapping.
4. Across all features belonging to the same example, keep the highest-scoring valid
   span as the final prediction.


In [ ]:
def postprocess_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {ex["id"]: i for i, ex in enumerate(examples)}
    features_per_example = collections.defaultdict(list)
    for i, feature_example_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_example_id]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        context = example["context"]

        valid_answers = []
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-n_best_size:][::-1]
            end_indexes = np.argsort(end_logits)[-n_best_size:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue
                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char],
                    })

        if valid_answers:
            best_answer = max(valid_answers, key=lambda x: x["score"])
        else:
            best_answer = {"text": "", "score": 0.0}

        predictions[example["id"]] = best_answer["text"]

    return predictions


In [ ]:
raw_predictions = trainer.predict(validation_dataset_for_model)

final_predictions = postprocess_predictions(
    raw_datasets["validation"], validation_dataset, raw_predictions.predictions
)


## 6. Evaluate: Exact Match and F1

In [ ]:
metric = evaluate.load("squad")

formatted_predictions = [
    {"id": k, "prediction_text": v} for k, v in final_predictions.items()
]
references = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in raw_datasets["validation"]
]

results = metric.compute(predictions=formatted_predictions, references=references)
print(results)


## 7. Error analysis

Look at the lowest-scoring predictions to see where the model struggles: very long
contexts, oddly phrased questions, or spans that are close but not an exact match
(e.g. missing/extra articles like "the" or trailing punctuation).


In [ ]:
def f1_score_single(prediction, ground_truth):
    pred_tokens = prediction.lower().split()
    gt_tokens = ground_truth.lower().split()
    common = collections.Counter(pred_tokens) & collections.Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens) if pred_tokens else 0.0
    recall = num_same / len(gt_tokens) if gt_tokens else 0.0
    return 2 * precision * recall / (precision + recall)

scored = []
for ex in raw_datasets["validation"]:
    pred = final_predictions[ex["id"]]
    gold = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    scored.append((f1_score_single(pred, gold), ex, pred, gold))

scored.sort(key=lambda x: x[0])

print("Ten worst predictions:\n")
for f1, ex, pred, gold in scored[:10]:
    print(f"F1={f1:.2f} | context_len={len(ex['context'])} chars")
    print("  Q:", ex["question"])
    print("  gold:", repr(gold))
    print("  pred:", repr(pred))
    print("-" * 80)


**What to write up here (based on what you actually observe when you run this):**
note whether the worst cases cluster around long contexts, multi-part or comparison
questions, answers that are numbers/dates, or predictions that are near-misses (right
general area, wrong exact boundary).


## 8. Save the fine-tuned model + tokenizer for the Streamlit app

In [ ]:
SAVE_DIR = "qa_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)


In [ ]:
# In Colab, zip the saved model folder and download it, then place it next to
# app.py before running `streamlit run app.py` (locally or e.g. via ngrok/Colab).
import shutil
shutil.make_archive("qa_model", "zip", SAVE_DIR)
print("Created qa_model.zip")

from google.colab import files
files.download("qa_model.zip")


### Sanity check: run the saved model with the same interface the Streamlit app uses

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model=SAVE_DIR, tokenizer=SAVE_DIR,
                        device=0 if torch.cuda.is_available() else -1)

sample_context = (
    "The Amazon rainforest covers much of northwestern Brazil and extends into "
    "Colombia, Peru and other South American countries. It is the largest "
    "tropical rainforest in the world, spanning about 5.5 million square kilometers."
)
sample_question = "How large is the Amazon rainforest?"

result = qa_pipeline(question=sample_question, context=sample_context)
print(result)
